# QMI2D validation — physical Dalitz bins

This notebook uses exact kinematic limits for the bin edges and an explicit physical-bin mask. Active cells are those that intersect the physical folded Dalitz region; therefore bins extend all the way to the kinematic endpoint even when only a small fraction of the last cell is physical.

In [ ]:
import numpy as np
import jax
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from dalitzplotfitter import (
    DalitzAmplitude, DecayChannel, DecayModel, QMI2D, RealImag,
    dalitz_s13_limits, enable_x64, physical_bin_mask, weighted_resample,
)
enable_x64()

channel = DecayChannel("D_s+", ("pi-", "pi+", "pi+"))
m1, m2, m3 = channel.daughter_masses
smin = (m1 + m2)**2
smax = (channel.parent_mass - m3)**2
edges = np.linspace(smin, smax, 9)
active_mask = physical_bin_mask(
    tuple(edges), tuple(edges),
    mother_mass=channel.parent_mass, masses=channel.daughter_masses,
    folded=True, samples_per_bin=257,
)
print("exact s range:", smin, smax)
print("active folded bins:", sum(sum(row) for row in active_mask))


## Physical boundary and active bins

In [ ]:
support = np.linspace(smin, smax, 3000)
low, high = dalitz_s13_limits(
    support, mother_mass=channel.parent_mass, masses=channel.daughter_masses
)
low, high = np.asarray(low), np.asarray(high)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(support, high, label="Dalitz boundary")
ax.plot(support, low)
ax.plot([smin, smax], [smin, smax], linestyle="--", label=r"$s_{12}=s_{13}$")
for i, row in enumerate(active_mask):
    for j, active in enumerate(row):
        if not active:
            continue
        ax.add_patch(Rectangle(
            (edges[i], edges[j]), edges[i+1]-edges[i], edges[j+1]-edges[j],
            fill=False, linewidth=1.2,
        ))
        ax.text(0.5*(edges[i]+edges[i+1]), 0.5*(edges[j]+edges[j+1]),
                f"{i},{j}", ha="center", va="center", fontsize=7)
ax.set_xlim(smin, smax); ax.set_ylim(smin, smax)
ax.set_aspect("equal", adjustable="box")
ax.set(xlabel=r"$s_{low}$ [GeV$^2$]", ylabel=r"$s_{high}$ [GeV$^2$]",
       title="Folded QMI2D bins intersecting the physical Dalitz region")
ax.legend(); plt.show()


## Same physical field with none, linear and cubic interpolation

In [ ]:
centers = 0.5*(edges[:-1] + edges[1:])
xx, yy = np.meshgrid(centers, centers, indexing="ij")
magnitudes = 1.0 + 0.8*np.exp(-((xx-0.9)**2 + (yy-1.8)**2)/0.35) + 0.25*np.sin(1.5*xx)
phases = 0.4 + 1.1*xx - 0.55*yy + 0.35*np.sin(2.0*yy)

def make_field(mode):
    return QMI2D(
        s12_edges=tuple(edges), s13_edges=tuple(edges),
        magnitudes=tuple(tuple(float(v) for v in row) for row in magnitudes),
        phases=tuple(tuple(float(v) for v in row) for row in phases),
        interpolation=mode, folded=True, active_mask=active_mask,
    )

base_model = DecayModel(
    channel, [DalitzAmplitude("qmi2d", make_field("none"), RealImag(1.0,0.0))],
    normalization_resolution=220,
)
grid = base_model.normalization_sample
data = grid.as_dict()
fig, axes = plt.subplots(3,2,figsize=(12,15),constrained_layout=True)
for row, mode in enumerate(("none","linear","cubic")):
    mag, phase = make_field(mode).interpolated_magnitude_phase(data)
    x = np.minimum(np.asarray(grid.s12), np.asarray(grid.s13))
    y = np.maximum(np.asarray(grid.s12), np.asarray(grid.s13))
    sc0=axes[row,0].scatter(x,y,c=np.asarray(mag),s=3); fig.colorbar(sc0,ax=axes[row,0],label="magnitude")
    sc1=axes[row,1].scatter(x,y,c=np.asarray(phase),s=3); fig.colorbar(sc1,ax=axes[row,1],label="phase [rad]")
    for ax in axes[row]: ax.set_xlim(smin,smax); ax.set_ylim(smin,smax); ax.set_aspect("equal",adjustable="box")
    axes[row,0].set_title(f"{mode}: magnitude"); axes[row,1].set_title(f"{mode}: phase")
plt.show()


## Cubic QMI2D toy

In [ ]:
model = DecayModel(
    channel, [DalitzAmplitude("qmi2d", make_field("cubic"), RealImag(1.0,0.0))],
    normalization_resolution=350,
)
N_POOL, N_TOY = 300_000, 50_000
pool = model.generate_phase_space(N_POOL, seed=6110)
target = pool.weights * model.intensity(pool.as_dict())
toy = weighted_resample(jax.random.key(6111), pool, target, N_TOY, replace=True)
fig, ax = plt.subplots(figsize=(7,6))
h=ax.hist2d(np.asarray(toy.s12),np.asarray(toy.s13),bins=100)
fig.colorbar(h[3],ax=ax,label="events")
ax.set(xlabel=r"$s_{12}$ [GeV$^2$]",ylabel=r"$s_{13}$ [GeV$^2$]",title=r"$D_s^+\to\pi^-\pi^+\pi^+$ QMI2D toy")
plt.show()
